This notebook provides an example of RAG using JPQ.

### Dependencies

In [1]:
%%bash

pip install -q transformers==4.57.1 vllm==0.8.* faiss-cpu datasets pyterrier_dr[jpq] pyterrier_rag

# for Colab
pip uninstall -y tensorflow torchcodec

### Loading a JPQ Index

In [2]:
from functools import cache

import pyterrier_dr as ptd
import pyterrier_dr.jpq
import pyterrier as pt
import pyterrier_rag as ptr
import pandas as pd
import datasets

2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.
2026-07-17 13:41:08 INFO     Using default tokenizer.


In [3]:
index = pyterrier_dr.jpq.JPQIndex.from_hf("jpq-repro/nq-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__")

https://huggingface.co/datasets/jpq-repro/nq-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn_…

extracting codes.f4 [1.9 GB]
extracting config.json [618 B]
extracting docnos.npids [207 B]
extracting model.safetensors [417.7 MB]
extracting opq.f4 [2.2 MB]
extracting pt_meta.json [107 B]
extracting subvecs.f4 [768.0 KB]


In [4]:
!ls -d {index.path}
!ls -lh {index.path}

/root/.pyterrier/artifacts/fd1ff7f570938e907814aed4afac73dc9a59850e64f0a9b13686794b93d16588
total 2.3G
-rw-r--r--. 1 root root 1.9G Jul 17 13:43 codes.f4
-rw-r--r--. 1 root root  618 Jul 17 13:43 config.json
-rw-r--r--. 1 root root  207 Jul 17 13:43 docnos.npids
-rw-r--r--. 1 root root 418M Jul 17 13:44 model.safetensors
-rw-r--r--. 1 root root 2.3M Jul 17 13:44 opq.f4
-rw-r--r--. 1 root root  107 Jul 17 13:44 pt_meta.json
-rw-r--r--. 1 root root 768K Jul 17 13:44 subvecs.f4



You should see the following files:
- `subvecs.f4`: the centroid embeddings (768KB)
- `opq.f4`: the OPQ rotation matrix (2.3MB)
- `codes.f4`: the codebook assigning documents to centroids (1.9GB)

**Note**: The jointly trained query encoder is saved under the same directory; we will load this checkpoint later for retrieval

In [5]:
len(index)

21015324

### NQ Dataset

Since NQ test split is not available through [ir_datasets](https://ir-datasets.com/natural-questions.html), here we implement an adapter to load it from [FlashRAG](https://huggingface.co/datasets/RUC-NLPIR/FlashRAG_datasets).

In [6]:
@cache
def _load(split: str) -> datasets.Dataset:
    return datasets.load_dataset("RUC-NLPIR/FlashRAG_datasets", "nq", split=split)

def get_topics(split: str) -> pd.DataFrame:
    dataset = _load(split)
    df = dataset.to_pandas().drop("golden_answers", axis=1)
    return df.rename({"id": "qid", "question": "query"}, axis=1)

def get_answers(split: str) -> pd.DataFrame:
    dataset = _load(split)
    data = []
    index = []
    for idx, item in enumerate(dataset):
        qid = item["id"]
        gold_answers = item["golden_answers"]
        for gold_answer in gold_answers:
            data.append([qid, gold_answer])
            index.append(idx)
    
    return pd.DataFrame(data, index=index, columns=["qid", "gold_answer"])

### Prompt & LLM Reader

Here we define a simple prompt and instantiate an LLM reader using [pyterrier_rag](https://github.com/terrierteam/pyterrier_rag/tree/main).

PS: The default configuration works on an NVIDIA 3090 GPU. You may adjust the vLLM configuration based on your hardware.

In [7]:
from transformers import PreTrainedTokenizerBase
from pyterrier_rag.readers import Reader
from pyterrier_rag.prompt import Concatenator, PromptTransformer
from pyterrier_rag.backend import TextGenerator, VLLMBackend, HuggingFaceBackend

# See Also: https://github.com/yuwvandy/KG-LLM-MDQA/blob/main/Pipeline/prompt.py
_PROMPT = """Given the following documents:
{qcontext}

Answer the following question: {query}

Your answer should be concise (no more than 6 words) and no explanation is needed.
Answer:"""


class PromptTransformerV2(PromptTransformer):
    def __init__(self, tokenizer: PreTrainedTokenizerBase, raw_prompt: str, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.raw_prompt = raw_prompt
        self.tokenizer = tokenizer

    def create_prompt(self, fields: dict) -> str:
        prompt = self.raw_prompt.format(**{k: fields[k] for k in self.input_fields})
        if self.tokenizer.chat_template:
            prompt = self.tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt},
                ],
                add_generation_prompt=True,
                tokenize=False,
            )

        return prompt


def format_doc(title: str, text: str) -> str:
    title = title.strip('"')
    text = text.removeprefix(title).lstrip()
    return f"Title: {title}\nText: {text}\n"

In [8]:
backend = ptr.VLLMBackend(
    "Qwen/Qwen2.5-3B-Instruct",
    model_args={
        "gpu_memory_utilization": 0.8,
        "max_num_seqs": 256,
        "max_num_batched_tokens": 2**16,
        # for older GPUs (e.g. NVIDIA T4)
        # "max_model_len": 8192,
        # "dtype": "half",
    },
    generation_args={
        "max_tokens": 32,
        "temperature": 0,
    },
)


INFO 07-17 13:44:13 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 07-17 13:44:13 [__init__.py:239] Automatically detected platform cuda.


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 07-17 13:44:30 [config.py:717] This model supports multiple tasks: {'reward', 'generate', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 07-17 13:44:30 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=65536.
INFO 07-17 13:44:40 [__init__.py:239] Automatically detected platform cuda.
INFO 07-17 13:44:44 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  4.09it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.64it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.79it/s]



INFO 07-17 13:44:51 [loader.py:458] Loading weights took 0.82 seconds
INFO 07-17 13:44:52 [gpu_model_runner.py:1347] Model loading took 5.7916 GiB and 6.892856 seconds
INFO 07-17 13:45:00 [backends.py:420] Using cache directory: /root/.cache/vllm/torch_compile_cache/cc9fb514dc/rank_0_0 for vLLM's torch.compile
INFO 07-17 13:45:00 [backends.py:430] Dynamo bytecode transform time: 8.38 s
INFO 07-17 13:45:07 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 6.319 s
INFO 07-17 13:45:08 [monitor.py:33] torch.compile takes 8.38 s in total
INFO 07-17 13:45:14 [kv_cache_utils.py:634] GPU KV cache size: 218,160 tokens
INFO 07-17 13:45:14 [kv_cache_utils.py:637] Maximum concurrency for 32,768 tokens per request: 6.66x
INFO 07-17 13:45:36 [gpu_model_runner.py:1686] Graph capturing finished in 22 secs, took 1.80 GiB
INFO 07-17 13:45:36 [core.py:159] init engine (profile, create kv cache, warmup model) took 44.01 seconds
INFO 07-17 13:45:36 [core_client.py:43

In [9]:
reader = Reader(
    backend=backend, 
    prompt=PromptTransformerV2(backend.model.get_tokenizer(), raw_prompt=_PROMPT, model_name_or_path=backend.model_id)
)
reader.backend.batch_size = 32
reader = Concatenator(in_fields=["title", "text"], intermediate_format=format_doc) >> reader

### RAG pipeline

We can then construct a RAG pipeline and evaluate it on the NQ test split.

In [10]:
encoder = pyterrier_dr.TctColBert.hnp()
encoder.model = encoder.model.from_pretrained(index.path).eval().to("cuda")

The `text_loader` below fetches document content (`title` and `text`) given retrieved document IDs.

In [11]:
text_loader = pt.Artifact.from_hf("pyterrier/ragwiki-terrier").text_loader(["docno", "title", "text"])

Java started (triggered by TerrierIndex.index_ref) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [12]:
jpq_rag = encoder.query_encoder(batch_size=256) >> index.retriever_pq(topk=5) >> text_loader >> reader

In [13]:
jpq_rag

(TctColBert('castorini/tct_colbert-v2-hnp-msmarco').query_encoder() >> JPQ-PQ >> <pyterrier.terrier._text_loader.TerrierTextLoader object at 0x7f86d97534f0> >> <pyterrier_rag.prompt._context_aggregation.Concatenator object at 0x7f85b8a07fd0> >> <pyterrier_rag.readers._base.Reader object at 0x7f85b8a06dd0>)

In [14]:
jpq_rag.search("who got the first nobel prize in physics")

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

,prompt,qid,query_0,qanswer
0,<|im_start|>system\nYou are a helpful assistan...,1,who got the first nobel prize in physics,Wilhelm Röntgen


In [15]:
pt.Experiment(
    [jpq_rag],
    get_topics("test"),
    get_answers("test"),
    [ptr.measures.F1, ptr.measures.EM],
    validate="ignore",
    names=["TCT-JPQ"],
)

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/26 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

,name,EM,F1
0,TCT-JPQ,0.263158,0.399522


Using TCT JPQ (`topk=5`), `Qwen2.5-3B-Instruct` gets 0.399 F1 and 0.263 EM. 